# 📊 Crypto Analysis - Análisis Técnico BTC + Portfolio

Notebook unificado que integra todo el análisis del proyecto original:

- **Análisis Técnico de Bitcoin** con TA-Lib (EMAs, RSI, MACD, Estocástico, ADX, ATR, OBV, Bollinger, MFI)
- **Sistema de Scoring** (+/-20) y recomendación (Compra/Venta/Neutral)
- **Gráfico interactivo** Plotly con 5 paneles
- **Portfolio Binance** (opcional, con API keys vía entorno)
- **Reporte HTML completo** auto-contenido


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import talib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import os, sys, json, webbrowser, warnings
import requests, hmac, hashlib, time
warnings.filterwarnings("ignore")
print("✅ Todos los imports cargados")


In [ ]:
TICKER = "BTC-USD"
TIMEFRAMES = {
    '1': ("1d", "2y", "1 DÍA"),
    '2': ("1h", "60d", "12 HORAS"),
    '3': ("1h", "60d", "6 HORAS"),
    '4': ("1h", "60d", "4 HORAS"),
    '5': ("1h", "7d", "1 HORA"),
    '6': ("30m", "7d", "30 MIN"),
    '7': ("15m", "5d", "15 MIN"),
    '8': ("5m", "5d", "5 MIN"),
    '9': ("1m", "1d", "1 MIN")
}
CHOICE = "6"
interval, periodo, timeframe = TIMEFRAMES[CHOICE]
print(f"⏱️ Timeframe: {timeframe} ({interval} / {periodo})")


In [ ]:
print(f"📥 Descargando {TICKER} ({timeframe})...")
data = yf.download(TICKER, period=periodo, interval=interval, progress=False)
data.columns = data.columns.droplevel(1)  # flatten MultiIndex
if data.empty:
    raise SystemExit("Error: No se pudieron descargar datos")
print(f"✅ {len(data)} velas | 📅 {data.index[0].strftime('%d/%m/%Y')} - {data.index[-1].strftime('%d/%m/%Y %H:%M')}")
print(f"💰 Precio: ${float(data['Close'].iloc[-1]):,.2f}")


In [ ]:
close = np.asarray(data['Close'].dropna().values, dtype=float).flatten()
high = np.asarray(data['High'].dropna().values, dtype=float).flatten()
low = np.asarray(data['Low'].dropna().values, dtype=float).flatten()
volume = np.asarray(data['Volume'].dropna().values, dtype=float).flatten()
open_prices = np.asarray(data['Open'].dropna().values, dtype=float).flatten()

ema_4 = talib.EMA(close, timeperiod=4)
ema_20 = talib.EMA(close, timeperiod=20)
ema_25 = talib.EMA(close, timeperiod=25)
ema_50 = talib.EMA(close, timeperiod=50)
ema_200 = talib.EMA(close, timeperiod=200)
rsi = talib.RSI(close, timeperiod=14)
rsi_5 = talib.RSI(close, timeperiod=5)
macd, signal, hist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
slowk, slowd = talib.STOCH(high, low, close, fastk_period=14, slowk_period=3, slowd_period=3)
adx = talib.ADX(high, low, close, timeperiod=14)
plus_di = talib.PLUS_DI(high, low, close, timeperiod=14)
minus_di = talib.MINUS_DI(high, low, close, timeperiod=14)
atr = talib.ATR(high, low, close, timeperiod=14)
obv = talib.OBV(close, volume)
obv_ema = talib.EMA(obv, timeperiod=20)
upper, middle, lower = talib.BBANDS(close, timeperiod=20, nbdevup=2, nbdevdn=2)
mfi = talib.MFI(high, low, close, volume, timeperiod=14)

print("✅ Indicadores calculados")


In [ ]:
precio = close[-1]
puntos = 0
senales = []

if not np.isnan(ema_20[-1]) and not np.isnan(ema_50[-1]) and not np.isnan(ema_200[-1]):
    if ema_20[-1] > ema_50[-1] and ema_50[-1] > ema_200[-1]:
        puntos += 3; tendencia = "🟢 ALCISTA"; senales.append("✅ EMAs alcistas (20>50>200)")
    elif ema_20[-1] < ema_50[-1] and ema_50[-1] < ema_200[-1]:
        puntos -= 3; tendencia = "🔴 BAJISTA"; senales.append("❌ EMAs bajistas (20<50<200)")
    else:
        tendencia = "⚪ NEUTRAL"; senales.append("⚠️ EMAs sin alineación clara")
else:
    tendencia = "⚪ NEUTRAL"; senales.append("⚠️ EMAs sin datos")

if precio > ema_4[-1]:
    puntos += 1; senales.append(f"✅ Precio > EMA4 (${ema_4[-1]:,.2f})")
else:
    puntos -= 1; senales.append(f"❌ Precio < EMA4 (${ema_4[-1]:,.2f})")

if not np.isnan(rsi[-1]):
    if rsi[-1] < 30:
        puntos += 2; senales.append(f"✅ RSI sobreventa ({rsi[-1]:.1f})")
    elif rsi[-1] > 70:
        puntos -= 2; senales.append(f"❌ RSI sobrecompra ({rsi[-1]:.1f})")
    elif rsi[-1] > 50:
        puntos += 1; senales.append(f"✅ RSI alcista ({rsi[-1]:.1f})")
    else:
        puntos -= 1; senales.append(f"❌ RSI bajista ({rsi[-1]:.1f})")

if not np.isnan(rsi_5[-1]):
    if rsi_5[-1] < 20:
        puntos += 1; senales.append(f"✅ RSI5 sobreventa extrema ({rsi_5[-1]:.1f})")
    elif rsi_5[-1] > 80:
        puntos -= 1; senales.append(f"❌ RSI5 sobrecompra extrema ({rsi_5[-1]:.1f})")

if not np.isnan(macd[-1]) and not np.isnan(signal[-1]):
    if macd[-1] > signal[-1]:
        puntos += 2; senales.append("✅ MACD > Signal")
    else:
        puntos -= 2; senales.append("❌ MACD < Signal")
    if not np.isnan(hist[-1]) and not np.isnan(hist[-2]) and not np.isnan(hist[-3]):
        if hist[-1] > hist[-2] > hist[-3]:
            puntos += 1; senales.append("✅ Histograma MACD creciendo")
        elif hist[-1] < hist[-2] < hist[-3]:
            puntos -= 1; senales.append("❌ Histograma MACD decreciendo")

if not np.isnan(adx[-1]) and adx[-1] > 25:
    if plus_di[-1] > minus_di[-1]:
        puntos += 2; senales.append(f"✅ Tendencia fuerte alcista (ADX {adx[-1]:.1f})")
    else:
        puntos -= 2; senales.append(f"❌ Tendencia fuerte bajista (ADX {adx[-1]:.1f})")
else:
    senales.append(f"⚠️ ADX débil ({adx[-1]:.1f})")

if not np.isnan(slowk[-1]):
    if slowk[-1] < 20 and slowd[-1] < 20:
        puntos += 1; senales.append("✅ Estocástico sobreventa")
    elif slowk[-1] > 80 and slowd[-1] > 80:
        puntos -= 1; senales.append("❌ Estocástico sobrecompra")
    if not np.isnan(slowk[-2]) and not np.isnan(slowd[-2]):
        if slowk[-1] > slowd[-1] and slowk[-2] < slowd[-2]:
            puntos += 1; senales.append("✅ Cruce alcista estocástico")
        elif slowk[-1] < slowd[-1] and slowk[-2] > slowd[-2]:
            puntos -= 1; senales.append("❌ Cruce bajista estocástico")

if not np.isnan(mfi[-1]):
    if mfi[-1] < 20:
        puntos += 1; senales.append(f"✅ MFI sobreventa ({mfi[-1]:.1f})")
    elif mfi[-1] > 80:
        puntos -= 1; senales.append(f"❌ MFI sobrecompra ({mfi[-1]:.1f})")

if not np.isnan(obv[-1]) and not np.isnan(obv_ema[-1]):
    if obv[-1] > obv_ema[-1]:
        puntos += 1; senales.append("✅ OBV > EMA (compra)")
    else:
        puntos -= 1; senales.append("❌ OBV < EMA (venta)")

if not np.isnan(lower[-1]):
    if precio < lower[-1]:
        puntos += 1; senales.append("✅ Precio bajo bandas")
    elif precio > upper[-1]:
        puntos -= 1; senales.append("❌ Precio sobre bandas")
    bb_width = (upper[-1] - lower[-1]) / middle[-1] * 100
    if bb_width < 2:
        senales.append(f"⚠️ Bollinger comprimidas ({bb_width:.2f}%)")

atr_val = atr[-1] if not np.isnan(atr[-1]) else 0

if puntos >= 8:
    rec = "🟢 COMPRA FUERTE"; color = "#00c853"
    tp = precio + (atr_val * 3); act_sl = precio - (atr_val * 1); sl = precio - (atr_val * 1.5)
elif puntos >= 4:
    rec = "🟢 COMPRA"; color = "#64dd17"
    tp = precio + (atr_val * 2.5); act_sl = precio - (atr_val * 1); sl = precio - (atr_val * 1.5)
elif puntos >= 1:
    rec = "🟡 COMPRA DÉBIL"; color = "#ffd600"
    tp = precio + (atr_val * 2); act_sl = precio - (atr_val * 0.8); sl = precio - (atr_val * 1.2)
elif puntos <= -8:
    rec = "🔴 VENTA FUERTE"; color = "#ff1744"
    tp = precio - (atr_val * 3); act_sl = precio + (atr_val * 1); sl = precio + (atr_val * 1.5)
elif puntos <= -4:
    rec = "🔴 VENTA"; color = "#ff5252"
    tp = precio - (atr_val * 2.5); act_sl = precio + (atr_val * 1); sl = precio + (atr_val * 1.5)
elif puntos <= -1:
    rec = "🟠 VENTA DÉBIL"; color = "#ff9100"
    tp = precio - (atr_val * 2); act_sl = precio + (atr_val * 0.8); sl = precio + (atr_val * 1.2)
else:
    rec = "⚪ NEUTRAL"; color = "#9e9e9e"; tp = act_sl = sl = None

rr = abs((tp - precio) / (precio - act_sl)) if tp and act_sl and (precio - act_sl) != 0 else 0
oco_type = rec.split()[-1] if tp else "NEUTRAL"

print(f"Puntos: {puntos:+d}/20")
print(f"Recomendación: {rec}")
if tp:
    print(f"TP: ${tp:.2f} | SL: ${sl:.2f} | R:R 1:{rr:.2f}")


In [ ]:
fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=('Precio + EMAs + BB', 'Volumen', 'RSI (14,5)', 'MACD', 'Estocástico'))

fig.add_trace(go.Candlestick(x=data.index, open=open_prices, high=high, low=low, close=close, name='BTC'), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=ema_4, mode='lines', name='EMA4', line=dict(color='purple', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=ema_20, mode='lines', name='EMA20', line=dict(color='yellow', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=ema_50, mode='lines', name='EMA50', line=dict(color='red', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=ema_200, mode='lines', name='EMA200', line=dict(color='white', width=1, dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=upper, mode='lines', name='BB Sup', line=dict(color='gray', width=1, dash='dot')), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=lower, mode='lines', name='BB Inf', line=dict(color='gray', width=1, dash='dot')), row=1, col=1)

fig.add_trace(go.Bar(x=data.index, y=volume/1e6, name='Vol (M)', marker_color='steelblue'), row=2, col=1)

fig.add_trace(go.Scatter(x=data.index, y=rsi, mode='lines', name='RSI 14', line=dict(color='purple')), row=3, col=1)
fig.add_trace(go.Scatter(x=data.index, y=rsi_5, mode='lines', name='RSI 5', line=dict(color='cyan', width=1)), row=3, col=1)
fig.add_hline(y=70, row=3, col=1, line_dash="dash", line_color="red", line_width=1)
fig.add_hline(y=30, row=3, col=1, line_dash="dash", line_color="green", line_width=1)

fig.add_trace(go.Scatter(x=data.index, y=macd, mode='lines', name='MACD', line=dict(color='blue')), row=4, col=1)
fig.add_trace(go.Scatter(x=data.index, y=signal, mode='lines', name='Signal', line=dict(color='red')), row=4, col=1)
hcolors = ['#00c853' if h > 0 else '#ff1744' for h in hist]
fig.add_trace(go.Bar(x=data.index, y=hist, name='Hist', marker_color=hcolors), row=4, col=1)

fig.add_trace(go.Scatter(x=data.index, y=slowk, mode='lines', name='%K', line=dict(color='blue')), row=5, col=1)
fig.add_trace(go.Scatter(x=data.index, y=slowd, mode='lines', name='%D', line=dict(color='red')), row=5, col=1)
fig.add_hline(y=80, row=5, col=1, line_dash="dash", line_color="red", line_width=1)
fig.add_hline(y=20, row=5, col=1, line_dash="dash", line_color="green", line_width=1)

fig.update_layout(title=f'BTC-USD | {timeframe} | {rec} | Score: {puntos:+d}/20',
                  height=1200, showlegend=False, template='plotly_dark', font=dict(size=10))
fig.show()


In [ ]:
class BinancePortfolio:
    def __init__(self, api_key=None, api_secret=None):
        self.api_key = api_key
        self.api_secret = api_secret
        self.base_url = 'https://api.binance.com'
        self.assets = []

    def _sign_request(self, params):
        qs = '&'.join([f"{k}={v}" for k, v in params.items()])
        return hmac.new(self.api_secret.encode('utf-8'), qs.encode('utf-8'), hashlib.sha256).hexdigest()

    def get_account_balance(self):
        if not self.api_key or not self.api_secret:
            print("API keys no configuradas"); return None
        ts = int(time.time() * 1000)
        params = {'timestamp': ts}
        sig = self._sign_request(params)
        headers = {'X-MBX-APIKEY': self.api_key}
        try:
            r = requests.get(f'{self.base_url}/api/v3/account', headers=headers, params={**params, 'signature': sig})
            r.raise_for_status(); return r.json()
        except Exception as e:
            print(f"Error API: {e}"); return None

    def get_all_prices(self):
        try:
            r = requests.get(f'{self.base_url}/api/v3/ticker/price')
            r.raise_for_status(); return {i['symbol']: float(i['price']) for i in r.json()}
        except: return {}

    def load_from_api(self, min_value_usd=1):
        account = self.get_account_balance()
        if not account: return False
        prices = self.get_all_prices()
        self.assets = []
        for bal in account.get('balances', []):
            asset = bal['asset']; total = float(bal['free']) + float(bal['locked'])
            if total <= 0: continue
            cp = 1.0 if asset in ('USDT','BUSD','USDC','TUSD') else prices.get(f'{asset}USDT', 0)
            val = total * cp
            if val >= min_value_usd:
                self.assets.append({'symbol': asset, 'quantity': total, 'free': float(bal['free']),
                    'locked': float(bal['locked']), 'current_price': cp, 'value_usd': val, 'buy_price': 0, 'source': 'api'})
        self.assets.sort(key=lambda x: x['value_usd'], reverse=True)
        print(f"{len(self.assets)} activos | Valor: ${sum(a['value_usd'] for a in self.assets):.2f}")
        return True

    def add_manual(self, symbol, quantity, buy_price, current_price=None):
        if current_price is None:
            current_price = self.get_all_prices().get(f'{symbol}USDT', 0)
        self.assets.append({'symbol': symbol.upper(), 'quantity': float(quantity), 'free': float(quantity),
            'locked': 0, 'current_price': float(current_price), 'value_usd': float(quantity)*float(current_price),
            'buy_price': float(buy_price), 'source': 'manual'})

    def update_buy_price(self, symbol, buy_price):
        for a in self.assets:
            if a['symbol'] == symbol.upper():
                a['buy_price'] = float(buy_price); print(f"Precio compra {symbol}: ${buy_price}"); return True
        return False

    def calculate_totals(self):
        tv = sum(a['value_usd'] for a in self.assets)
        tc = sum(a['quantity'] * a['buy_price'] for a in self.assets if a['buy_price'] > 0)
        return {'total_value': tv, 'total_cost': tc, 'total_pl': tv - tc,
                'total_pl_percent': (tv - tc) / tc * 100 if tc > 0 else 0,
                'assets_count': len(self.assets), 'assets_with_pl': sum(1 for a in self.assets if a['buy_price'] > 0)}

# --- Cargar ---
API_KEY = os.environ.get('BINANCE_API_KEY', '')
API_SECRET = os.environ.get('BINANCE_API_SECRET', '')
portfolio = None
if API_KEY and API_SECRET:
    print("Conectando con Binance...")
    portfolio = BinancePortfolio(API_KEY, API_SECRET)
    ok = portfolio.load_from_api(min_value_usd=1)
    if ok:
        portfolio.update_buy_price('BTC', 45000)
        portfolio.update_buy_price('ETH', 2500)
        portfolio.update_buy_price('BNB', 300)
        t = portfolio.calculate_totals()
        print(f"Valor: ${t['total_value']:.2f} | P&L: ${t['total_pl']:+.2f} ({t['total_pl_percent']:+.2f}%)")
else:
    print("Portfolio: BINANCE_API_KEY/BINANCE_API_SECRET no configuradas")


In [ ]:
def build_portfolio_html(pf):
    if not pf or not pf.assets:
        return "<div style='padding:20px;text-align:center;color:#888;'>Portfolio no disponible</div>"
    t = pf.calculate_totals()
    rows = ''
    for a in pf.assets:
        cost = a['quantity'] * a['buy_price'] if a['buy_price'] > 0 else 0
        pl = a['value_usd'] - cost
        pl_pct = (pl / cost * 100) if cost > 0 else 0
        pl_cls = 'positive' if pl >= 0 else 'negative'
        bp_str = f"${a['buy_price']:.2f}" if a['buy_price'] > 0 else '<span style="color:#999">N/A</span>'
        pl_str = f'<span class="{pl_cls}">${pl:.2f}</span>' if a['buy_price'] > 0 else '<span style="color:#999">N/A</span>'
        pct_str = f'<span class="{pl_cls}">{pl_pct:+.2f}%</span>' if a['buy_price'] > 0 else '<span style="color:#999">N/A</span>'
        rows += f'<tr><td><strong>{a["symbol"]}</strong></td><td>{a["quantity"]:.8f}</td><td>{bp_str}</td><td>${a["current_price"]:.8f}</td><td><strong>${a["value_usd"]:.2f}</strong></td><td>{pl_str}</td><td>{pct_str}</td></tr>'
    pl_cls_t = 'positive' if t['total_pl'] >= 0 else 'negative'
    return (''
    f'<div class="pf-summary">'
    f'<div class="pf-card"><h3>Valor Total</h3><div class="pf-amount">${t["total_value"]:.2f}</div></div>'
    f'<div class="pf-card"><h3>Activos</h3><div class="pf-amount">{t["assets_count"]}</div></div>'
    f'<div class="pf-card"><h3>Inversión</h3><div class="pf-amount">${t["total_cost"]:.2f}</div></div>'
    f'<div class="pf-card"><h3>G/P</h3><div class="pf-amount {pl_cls_t}">${t["total_pl"]:+.2f}</div><div class="{pl_cls_t}">{t["total_pl_percent"]:+.2f}%</div></div>'
    '</div>'
    f'<table><thead><tr><th>Activo</th><th>Cantidad</th><th>Precio Compra</th><th>Precio Actual</th><th>Valor</th><th>G/P ($)</th><th>G/P (%)</th></tr></thead><tbody>{rows}</tbody></table>'
    )

pf_html = build_portfolio_html(portfolio)

if tp:
    oco_items = (''
        f'<div class="oco-item"><div class="label">TP Limit</div><div class="value">${tp:,.2f}</div></div>'
        f'<div class="oco-item"><div class="label">Activador SL</div><div class="value">${act_sl:,.2f}</div></div>'
        f'<div class="oco-item"><div class="label">SL Limit</div><div class="value">${sl:,.2f}</div></div>'
        f'<div class="oco-item"><div class="label">Risk/Reward</div><div class="value">1:{rr:.2f}</div></div>'
    )
else:
    oco_items = '<div class="oco-item" style="grid-column:1/-1;"><div class="label">Estado</div><div class="value">⚠️ NO OPERAR</div></div>'

senales_html = ''.join(f'<div class="signal-item">{i+1}. {s}</div>' for i, s in enumerate(senales))

ind_rows = (''
    f'<tr><td>EMA 4 / 20 / 50 / 200</td><td>${ema_4[-1]:,.2f} / ${ema_20[-1]:,.2f} / ${ema_50[-1]:,.2f} / ${ema_200[-1]:,.2f}</td></tr>'
    f'<tr><td>RSI (14)</td><td>{rsi[-1]:.2f}</td></tr>'
    f'<tr><td>RSI (5)</td><td>{rsi_5[-1]:.2f}</td></tr>'
    f'<tr><td>MACD / Signal</td><td>{macd[-1]:.4f} / {signal[-1]:.4f}</td></tr>'
    f'<tr><td>ADX</td><td>{adx[-1]:.2f}</td></tr>'
    f'<tr><td>ATR</td><td>${atr[-1]:,.2f}</td></tr>'
    f'<tr><td>Estocástico %K / %D</td><td>{slowk[-1]:.2f} / {slowd[-1]:.2f}</td></tr>'
    f'<tr><td>MFI</td><td>{mfi[-1]:.2f}</td></tr>'
    f'<tr><td>Bollinger Sup / Inf</td><td>${upper[-1]:,.2f} / ${lower[-1]:,.2f}</td></tr>'
)

html_output = (''
"<!DOCTYPE html>\n"
"<html lang=\"es\">\n"
"<head>\n"
"<meta charset=\"UTF-8\">\n"
"<meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">\n"
f'<title>Crypto Analysis | {timeframe}</title>\n'
"<style>\n"
"* { margin:0; padding:0; box-sizing:border-box; }\n"
"body { font-family:'Segoe UI',sans-serif; background:#0a0a0a; color:#fff; padding:20px; }\n"
f'.container {{ max-width:1600px; margin:0 auto; }}\n'
f'.header {{ background:linear-gradient(135deg,#667eea,#764ba2); padding:30px; border-radius:12px; margin-bottom:25px; text-align:center; }}\n'
"h1 { margin-bottom:10px; }\n"
f'.price {{ font-size:42px; font-weight:bold; margin:15px 0; }}\n'
f'.oco-box {{ background:#1a1a1a; border:3px solid {color}; border-radius:12px; padding:25px; margin:25px 0; }}\n'
f'.oco-title {{ font-size:26px; font-weight:bold; color:{color}; text-align:center; margin-bottom:20px; }}\n'
".oco-grid { display:grid; grid-template-columns:repeat(auto-fit,minmax(200px,1fr)); gap:15px; }\n"
f'.oco-item {{ background:#252525; padding:15px; border-radius:8px; border-left:4px solid {color}; }}\n'
".label { font-size:11px; color:#888; text-transform:uppercase; letter-spacing:1px; }\n"
".value { font-size:22px; font-weight:bold; color:#fff; }\n"
".rec-box { text-align:center; padding:15px; background:#252525; border-radius:8px; margin-top:15px; }\n"
f'.rec {{ color:{color}; font-size:28px; font-weight:bold; }}\n'
".signals { background:#1a1a1a; padding:20px; border-radius:12px; margin:25px 0; }\n"
".signal-item { padding:10px; margin:6px 0; background:#252525; border-radius:6px; border-left:3px solid #667eea; }\n"
".chart-container { background:#1a1a1a; padding:15px; border-radius:12px; margin:25px 0; }\n"
".section-title { color:#667eea; margin-bottom:15px; }\n"
".positive { color:#00c853; }\n"
".negative { color:#ff1744; }\n"
".pf-summary { display:grid; grid-template-columns:repeat(auto-fit,minmax(200px,1fr)); gap:20px; margin-bottom:25px; }\n"
".pf-card { background:#252525; padding:20px; border-radius:12px; text-align:center; }\n"
".pf-card h3 { color:#888; font-size:12px; text-transform:uppercase; }\n"
".pf-amount { font-size:28px; font-weight:bold; color:#f7931a; }\n"
"table { width:100%; border-collapse:collapse; margin:15px 0; }\n"
"th { background:#f7931a; color:#fff; padding:12px; text-align:left; }\n"
"td { padding:12px; border-bottom:1px solid #333; }\n"
"tr:hover { background:#1a1a1a; }\n"
".timestamp { text-align:center; color:#666; margin-top:30px; }\n"
"@media (max-width:768px) { .oco-grid { grid-template-columns:1fr 1fr; } }\n"
"</style>\n"
"</head>\n"
"<body>\n"
"<div class=\"container\">\n"
"<div class=\"header\">\n"
"<h1>📊 CRYPTO ANALYSIS</h1>\n"
f'<div style="font-size:18px;">⏱️ {timeframe} | {data.index[-1].strftime('%d/%m/%Y %H:%M')}</div>\n'
f'<div class="price">${precio:,.2f}</div>\n'
f'<div style="font-size:16px;opacity:0.8;">{tendencia} | Puntuación: {puntos:+d}/20</div>\n'
"</div>\n"
"<div class=\"oco-box\">\n"
f'<div class="oco-title">🎯 ORDEN OCO ({oco_type})</div>\n'
"<div class=\"oco-grid\">\n"
f'<div class="oco-item"><div class="label">Precio Actual</div><div class="value">${precio:,.2f}</div></div>\n'
f'{oco_items}\n'
"</div>\n"
f'<div class="rec-box"><div class="rec">{rec}</div></div>\n'
"</div>\n"
f'<div class="signals">\n<h2 class="section-title">🔍 Señales ({len(senales)})</h2>\n{senales_html}\n</div>\n'
f'<div class="signals">\n<h2 class="section-title">📈 Indicadores</h2>\n<table>\n<tr><th>Indicador</th><th>Valor</th></tr>{ind_rows}</table>\n</div>\n'
f'<div class="signals">\n<h2 class="section-title">🔸 Portfolio Binance</h2>\n{pf_html}\n</div>\n'
f'<div class="chart-container">\n<h2 class="section-title">📈 Gráfico</h2>\n{fig.to_html(include_plotlyjs="cdn", full_html=False)}\n</div>\n'
f'<div class="timestamp">Generado el {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}</div>\n'
"</div>\n"
"</body>\n"
"</html>"
)

output_file = 'crypto_analysis.html'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(html_output)
print(f"✅ Reporte HTML: {output_file}")
print(f"📁 Tamaño: {os.path.getsize(output_file):,} bytes")


In [ ]:
try:
    webbrowser.open(f'file://{os.path.abspath(output_file)}')
    print("Navegador abierto")
except:
    print(f"Abre manualmente: {output_file}")
